### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model=init_chat_model(model="gpt-5")
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high']}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000291CD51E510>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000291CD51EF9

In [2]:
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots “talk” because they are exceptional vocal learners. Unlike most animals, they can hear a sound, form an internal model of it, and deliberately reproduce it with their own voice. That skill evolved for parrot-to-parrot communication, and in captivity it gets applied to human speech.\n\nKey points:\n- Social reason for learning: Wild parrots live in complex, long‑lived social groups. They learn and modify calls to match their flock’s “dialect,” keep track of partners, and broadcast individual identity. Juveniles learn from tutors, and many species keep learning throughout life, so their vocal systems stay flexible.\n- How they make human-like sounds: Parrots don’t have vocal cords. They use a bird organ called the syrinx, plus very fine control of their tongue, beak, and throat to shape sound. This lets them approximate our vowels and many consonants well enough for us to recognize words.\n- Brain wiring: Parrots have specialized forebrain circuits for linking 

In [3]:
response.content

'Parrots “talk” because they are exceptional vocal learners. Unlike most animals, they can hear a sound, form an internal model of it, and deliberately reproduce it with their own voice. That skill evolved for parrot-to-parrot communication, and in captivity it gets applied to human speech.\n\nKey points:\n- Social reason for learning: Wild parrots live in complex, long‑lived social groups. They learn and modify calls to match their flock’s “dialect,” keep track of partners, and broadcast individual identity. Juveniles learn from tutors, and many species keep learning throughout life, so their vocal systems stay flexible.\n- How they make human-like sounds: Parrots don’t have vocal cords. They use a bird organ called the syrinx, plus very fine control of their tongue, beak, and throat to shape sound. This lets them approximate our vowels and many consonants well enough for us to recognize words.\n- Brain wiring: Parrots have specialized forebrain circuits for linking sounds they hear t

In [4]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_tools = model.bind_tools([get_weather])

In [8]:
response = model_with_tools.invoke("What is the weather like in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 136, 'total_tokens': 224, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EOL6PColRjfwHqNt7txhRRhpohg72', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a0a4b4-a3d5-74b0-b395-03ccca0a52c1-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_kxt0lsGPoGJNmjDid3BvmtvV', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 136, 'output_tokens': 88, 'total_tokens': 224, 'input_token_details': {'audio': 0, 'cache_read': 0},

### Tool Execution Loops

In [9]:
# Step 1: Model generates tool calls
messages = [{"role" : "user", "content" : "What is the weather like in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    
#step 3: Pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It’s sunny in New York with a temperature of about 25°C.


In [10]:
messages

[{'role': 'user', 'content': 'What is the weather like in New York?'},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 136, 'total_tokens': 288, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EOLCOsSp5rSzcjRAbliALMR3lBPjo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a4ba-5004-7ed3-b506-b12ddcae5e1e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_K20JcvUfoUVxu9wWnYRlzfD1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 136, 'outp